<a href="https://colab.research.google.com/github/Jirehcode/Codveda-ML-tasks/blob/main/task-2-web-scraping/packaging_specification_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import threading
import time
from http.server import HTTPServer, SimpleHTTPRequestHandler
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import pandas as pd
import requests


In [ ]:
port = 888
BASE_URL = F"HTTP://127.0.0.1:{port}/"
PAGE_1_HTML = """
<!DOCTYPE html>
<html>
<body>
    <h1>Page 1 Content</h1>
    <div class="product-card">
        <h3 class="product-title"><a href="/product1">Product A</a></h3>
        <p class="category">Category X</p>
        <table class="tech-specs">
            <tr><td>Structure</td><td>PET/AL/LLDPE</td></tr>
            <tr><td>Thickness</td><td>120 um</td></tr>
            <tr><td>Bond Strength</td><td>5.2 N/15mm</td></tr>
            <tr><td>Seal Strength</td><td>6.0 N/15mm</td></tr>
            <tr><td>COF</td><td>0.3 Kinetic</td></tr>
        </table>
    </div>
    <div class="product-card">
        <h3 class="product-title"><a href="/product2">Product B</a></h3>
        <p class="category">Category Y</p>
        <table class="tech-specs">
            <tr><td>Structure</td><td>OPP/AL</td></tr>
            <tr><td>Thickness</td><td>80 um</td></tr>
            <tr><td>Bond Strength</td><td>4.8 N/15mm</td></tr>
            <tr><td>Seal Strength</td><td>5.5 N/15mm</td></tr>
            <tr><td>COF</td><td>0.4 Kinetic</td></tr>
        </table>
    </div>
    <li class="next"><a href="/page2">Next Page</a></li>
</body>
</html>
"""

PAGE_2_HTML = """
<!DOCTYPE html>
<html>
<body>
    <h1>Page 2 Content</h1>
    <div class="product-card">
        <h3 class="product-title"><a href="/product3">Product C</a></h3>
        <p class="category">Category Z</p>
        <table class="tech-specs">
            <tr><td>Structure</td><td>PE/EVOH/PE</td></tr>
            <tr><td>Thickness</td><td>150 um</td></tr>
            <tr><td>Bond Strength</td><td>6.1 N/15mm</td></tr>
            <tr><td>Seal Strength</td><td>6.8 N/15mm</td></tr>
            <tr><td>COF</td><td>0.2 Kinetic</td></tr>
        </table>
    </div>
    <div class="product-card">
        <h3 class="product-title"><a href="/product4">Product D</a></h3>
        <p class="category">Category X</p>
        <table class="tech-specs">
            <tr><td>Structure</td><td>PAPER/AL/PE</td></tr>
            <tr><td>Thickness</td><td>100 um</td></tr>
            <tr><td>Bond Strength</td><td>5.0 N/15mm</td></tr>
            <tr><td>Seal Strength</td><td>5.2 N/15mm</td></tr>
            <tr><td>COF</td><td>0.35 Kinetic</td></tr>
        </table>
    </div>
</body>
</html>
"""

def do_GET(self):
  self.send_response(200)
  self.send_header("content_type", "text/html")
  self.end_headers()
  if "page2" in self.path:
    self.wfile.write(PAGE_2_HTML.encode("utf-8"))
  else:
    self.wfile.write(PAGE_1_HTML.encode("utf-8"))

def log_message(self, format, *args):
  return

In [ ]:
def extract_numeric(text):

    try:
        return float(re.findall(r'\d+\.?\d*', text)[0])
    except (IndexError, ValueError):
        return None

def parse_card_specs(card_soup):
  specs = {}
  rows = card_soup.select("table.tech-specs tr")
  for row in rows:
    cols = row.find_all("td")
    if len(cols) == 0:
      continue

    label = cols[0].get_text().strip().lower()
    val_text = cols[1].get_text().strip()

    if "structure" in label:
      specs["structure"] = val_text
    elif "thickness" in label:
      specs["thickness_um"] = extract_numeric(val_text)
    elif "bond strength" in label:
      specs["bond_strength_N15mm"] = extract_numeric(val_text)
    elif "seal strength" in label:
      specs["seal_strength_N15mm"] = extract_numeric(val_text)
    elif "cof" in label:
      specs["cof_kinetic"] = extract_numeric(val_text)

  return specs

In [ ]:
from itertools import product
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;image/avif,image/webp,image/apng,*/*;q=0.8",
})

def scrape_website(start_url, max_pages_to_scrape):
  scraped_records = []
  current_url = start_url
  pages_scraped = 0
  max_pages = max_pages_to_scrape

  while current_url and pages_scraped < max_pages:
    print(f"[+] Scraping page {pages_scraped + 1} : {current_url}")

    try:
      response = session.get(current_url, timeout=10)
      response.raise_for_status()
    except requests.RequestException as err:
      print(f"[-] HTTP Error accessing {current_url}: {err}")
      break

    soup = bs4.BeautifulSoup(response.text, "html.parser")
    product_cards = soup.select("div.product-card")

    for card in product_cards:
      title_elem = card.select_one("h3.product-title a")
      product_name = (
          title_elem.get_text().strip() if title_elem else "N/A"
      )
      rel_link = title_elem["href"] if title_elem else "N/A"
      product_url = urljoin(current_url, rel_link)

      category_elem = card.select_one(".category")

      product_category = (
          category_elem.get_text().strip() if category_elem else "N/A"
      )

      tech_specs = parse_card_specs(card)

      record = {
          "product_name": product_name,
          "product_url": product_url,
          "product_category": product_category,
          "structure": tech_specs.get("structure"),
          "thickness_um": tech_specs.get("thickness_um"),
          "bond_strength_N15mm": tech_specs.get("bond_strength_N15mm"),
          "seal_strength_N15mm": tech_specs.get("seal_strength_N15mm"),
          "cof_kinetic": tech_specs.get("cof_kinetic"),

      }
      scraped_records.append(record)

    pages_scraped += 1

    next_button = soup.select_one("li.next a")
    if next_button and next_button.has_attr("href"):
      current_url = urljoin(current_url, next_button["href"])
      time.sleep(1)
    else:
      current_url = None

  return scraped_records


In [ ]:
from IPython.utils import encoding

class MyHandler(SimpleHTTPRequestHandler):
    def do_GET(self):
        do_GET(self)

    def log_message(self, format, *args):

        log_message(self, format, *args)


def start_server():
    global httpd
    server_address = ('', port)
    httpd = HTTPServer(server_address, MyHandler)
    httpd.serve_forever()

print(f"Starting server on port {port}...")
server_thread = threading.Thread(target=start_server)
server_thread.daemon = True
server_thread.start()

time.sleep(2)

start_url = BASE_URL
max_pages_to_scrape = 2
print(f"\nScraping data from {start_url} for up to {max_pages_to_scrape} pages...")
scraped_records = scrape_website(start_url, max_pages_to_scrape)

df = pd.DataFrame(scraped_records)

numeric_cols = [
  "thickness_um",
  "bond_strength_N15mm",
  "seal_strength_N15mm",
  "cof_kinetic",
]

for col in numeric_cols:
  df[col] = pd.to_numeric(df[col], errors="coerce")


filename = "scraped_data.csv"

df.to_csv(filename, index=False, encoding="utf-8")
print(f"\nData successfully saved to '{filename}' ({len(df)} records)")
print("\nDataset Summary for Exploratory Analysis:")
print(df.to_string(index=False))


Starting server on port 888...

Scraping data from HTTP://127.0.0.1:888/ for up to 2 pages...
[+] Scraping page 1 : HTTP://127.0.0.1:888/
[+] Scraping page 2 : http://127.0.0.1:888/page2

Data successfully saved to 'scraped_data.csv' (4 records)

Dataset Summary for Exploratory Analysis:
product_name                   product_url product_category    structure  thickness_um  bond_strength_N15mm  seal_strength_N15mm  cof_kinetic
   Product A http://127.0.0.1:888/product1       Category X PET/AL/LLDPE         120.0                  5.2                  6.0         0.30
   Product B http://127.0.0.1:888/product2       Category Y       OPP/AL          80.0                  4.8                  5.5         0.40
   Product C http://127.0.0.1:888/product3       Category Z   PE/EVOH/PE         150.0                  6.1                  6.8         0.20
   Product D http://127.0.0.1:888/product4       Category X  PAPER/AL/PE         100.0                  5.0                  5.2         0.35


In [ ]:
try:
  print("[*] Launching flexible packaging web scraper...")False)
  print(f"[*] Scraping and data processing complete. Records in DataFrame: {len(df) if 'df' in globals() else 'N/A'}")

finally:
  if 'httpd' in globals() and httpd:
    print("[*] Shutting down HTTP server...")
    httpd.shutdown()
  print("\n[*] Server shutdown complete.")

[*] Launching flexible packaging web scraper...
[*] Scraping and data processing complete. Records in DataFrame: 4
[*] Shutting down HTTP server...

[*] Server shutdown complete.


In [ ]:
from google.colab import files
files.download("scraped_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>